# MIMIC preprocessing for irregular time-series + optional text

This notebook rebuilds the MIMIC preprocessing pipeline with these fixes:

- removes the old `elapsed_days > 50` long-stay filter
- keeps the **structured irregular time-series cohort** primary
- treats text as **optional** rather than a hard inclusion criterion
- supports **task-specific cohorts** for:
  - mortality
  - ICU readmission
- uses an **early observation window** to reduce leakage
- writes patient folders with:
  - `time_series.csv`
  - `text.csv` (empty if no notes)
- copies task-specific folders into:
  - `processed_mortality`
  - `processed_icu_readmit`

Update the paths in the configuration cell if needed, then run top to bottom.

In [1]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Install if needed in Colab
# !pip -q install pandas numpy pyarrow

In [2]:
import os
import gc
import shutil
from pathlib import Path
import numpy as np
import pandas as pd

## 1. Configuration

In [3]:
# =========================
# USER CONFIGURATION
# =========================

MIMIC_ROOT = Path("/content/drive/MyDrive/mimic_graph_project/mimic-iv-3.1")
NOTE_ROOT  = Path("/content/drive/MyDrive/mimic_graph_project/note")
OUT_BASE   = Path("/content/drive/MyDrive/IMM-TSF/data/MIMIC_fixed")
RUNLOG_DIR = Path("/content/drive/MyDrive/IMM-TSF/run_logs")

RUNLOG_DIR.mkdir(parents=True, exist_ok=True)
OUT_BASE.mkdir(parents=True, exist_ok=True)

# Which note files to include if present
NOTE_FILES = [
    "radiology.csv.gz",
    "discharge.csv.gz",
]

# Observation windows
MORTALITY_OBS_HOURS = 24
READMIT_OBS_HOURS   = 48

# Minimum irregular-history rule
MIN_UNIQUE_TIMES = 8
MIN_OBS_VALUES = 25
MIN_DISTINCT_VARIABLES = 3

# Feature tables to use
USE_INPUTEVENTS = True
USE_LABEVENTS   = True
USE_CHARTEVENTS = False   # set True only if you have enough RAM / disk

# Optional row caps for very large tables while debugging; set to None for full run
DEBUG_NROWS_INPUTEVENTS = None
DEBUG_NROWS_LABEVENTS   = None
DEBUG_NROWS_CHARTEVENTS = None

print("MIMIC_ROOT:", MIMIC_ROOT)
print("NOTE_ROOT :", NOTE_ROOT)
print("OUT_BASE  :", OUT_BASE)

MIMIC_ROOT: /content/drive/MyDrive/mimic_graph_project/mimic-iv-3.1
NOTE_ROOT : /content/drive/MyDrive/mimic_graph_project/note
OUT_BASE  : /content/drive/MyDrive/IMM-TSF/data/MIMIC_fixed


## 2. Load admissions and create labels

In [4]:
adm = pd.read_csv(MIMIC_ROOT / "hosp" / "admissions.csv.gz", low_memory=False)
adm["hadm_id"] = pd.to_numeric(adm["hadm_id"], errors="coerce")
adm["subject_id"] = pd.to_numeric(adm["subject_id"], errors="coerce")
adm = adm.dropna(subset=["hadm_id", "subject_id"]).copy()
adm["hadm_id"] = adm["hadm_id"].astype(int)
adm["subject_id"] = adm["subject_id"].astype(int)

adm["admittime"] = pd.to_datetime(adm["admittime"], errors="coerce")
adm["dischtime"] = pd.to_datetime(adm["dischtime"], errors="coerce")

if "deathtime" in adm.columns:
    adm["deathtime"] = pd.to_datetime(adm["deathtime"], errors="coerce")

if "hospital_expire_flag" in adm.columns:
    adm["mortality_label"] = pd.to_numeric(adm["hospital_expire_flag"], errors="coerce").fillna(0).astype(int)
else:
    adm["mortality_label"] = adm["deathtime"].notna().astype(int)

# Simple hospital-level readmission label:
# positive if same subject has a later admission after discharge
adm = adm.sort_values(["subject_id", "admittime"]).copy()
adm["next_admittime"] = adm.groupby("subject_id")["admittime"].shift(-1)
adm["readmit_label"] = (
    adm["dischtime"].notna()
    & adm["next_admittime"].notna()
    & (adm["next_admittime"] > adm["dischtime"])
    & (adm["mortality_label"] == 0)
).astype(int)

label_df = adm[["hadm_id", "subject_id", "mortality_label", "readmit_label", "admittime", "dischtime"]].copy()
print("Admissions:", len(label_df))
print(label_df[["mortality_label", "readmit_label"]].mean())
label_df.head()

Admissions: 546028
mortality_label    0.021612
readmit_label      0.586860
dtype: float64


,hadm_id,subject_id,mortality_label,readmit_label,admittime,dischtime
0,22595853,10000032,0,1,2180-05-06 22:23:00,2180-05-07 17:15:00
1,22841357,10000032,0,1,2180-06-26 18:27:00,2180-06-27 18:49:00
3,29079034,10000032,0,1,2180-07-23 12:35:00,2180-07-25 17:55:00
2,25742920,10000032,0,0,2180-08-05 23:44:00,2180-08-07 17:50:00
4,25022803,10000068,0,0,2160-03-03 23:16:00,2160-03-04 06:26:00


## 3. Build irregular structured event table

In [5]:
def load_inputevents():
    path = MIMIC_ROOT / "icu" / "inputevents.csv.gz"
    if not path.exists():
        return None
    df = pd.read_csv(path, low_memory=False, nrows=DEBUG_NROWS_INPUTEVENTS)
    keep = [c for c in ["hadm_id", "starttime", "itemid", "amount"] if c in df.columns]
    df = df[keep].copy()
    df["hadm_id"] = pd.to_numeric(df["hadm_id"], errors="coerce")
    df = df.dropna(subset=["hadm_id"]).copy()
    df["hadm_id"] = df["hadm_id"].astype(int)
    df["time"] = pd.to_datetime(df["starttime"], errors="coerce")
    df["variable"] = "input_" + df["itemid"].astype(str)
    df["value"] = pd.to_numeric(df["amount"], errors="coerce")
    df = df[["hadm_id", "time", "variable", "value"]].dropna(subset=["time", "variable"])
    return df

def load_labevents():
    path = MIMIC_ROOT / "hosp" / "labevents.csv.gz"
    if not path.exists():
        return None
    df = pd.read_csv(path, low_memory=False, nrows=DEBUG_NROWS_LABEVENTS)
    keep = [c for c in ["hadm_id", "charttime", "itemid", "valuenum"] if c in df.columns]
    df = df[keep].copy()
    df["hadm_id"] = pd.to_numeric(df["hadm_id"], errors="coerce")
    df = df.dropna(subset=["hadm_id"]).copy()
    df["hadm_id"] = df["hadm_id"].astype(int)
    df["time"] = pd.to_datetime(df["charttime"], errors="coerce")
    df["variable"] = "lab_" + df["itemid"].astype(str)
    df["value"] = pd.to_numeric(df["valuenum"], errors="coerce")
    df = df[["hadm_id", "time", "variable", "value"]].dropna(subset=["time", "variable"])
    return df

def load_chartevents():
    path = MIMIC_ROOT / "icu" / "chartevents.csv.gz"
    if not path.exists():
        return None
    df = pd.read_csv(path, low_memory=False, nrows=DEBUG_NROWS_CHARTEVENTS)
    keep = [c for c in ["hadm_id", "charttime", "itemid", "valuenum"] if c in df.columns]
    df = df[keep].copy()
    df["hadm_id"] = pd.to_numeric(df["hadm_id"], errors="coerce")
    df = df.dropna(subset=["hadm_id"]).copy()
    df["hadm_id"] = df["hadm_id"].astype(int)
    df["time"] = pd.to_datetime(df["charttime"], errors="coerce")
    df["variable"] = "chart_" + df["itemid"].astype(str)
    df["value"] = pd.to_numeric(df["valuenum"], errors="coerce")
    df = df[["hadm_id", "time", "variable", "value"]].dropna(subset=["time", "variable"])
    return df

parts = []

if USE_INPUTEVENTS:
    x = load_inputevents()
    if x is not None:
        print("inputevents:", x.shape)
        parts.append(x)

if USE_LABEVENTS:
    x = load_labevents()
    if x is not None:
        print("labevents:", x.shape)
        parts.append(x)

if USE_CHARTEVENTS:
    x = load_chartevents()
    if x is not None:
        print("chartevents:", x.shape)
        parts.append(x)

if len(parts) == 0:
    raise RuntimeError("No structured tables loaded.")

ts_long = pd.concat(parts, ignore_index=True)
ts_long = ts_long.loc[ts_long["hadm_id"].isin(label_df["hadm_id"])].copy()
ts_long = ts_long.dropna(subset=["hadm_id", "time", "variable"]).copy()
ts_long["hadm_id"] = ts_long["hadm_id"].astype(int)

print("Long structured rows:", len(ts_long))
print("Unique hadm_id with structured data:", ts_long["hadm_id"].nunique())

del parts
gc.collect()

ts_long.head()

inputevents: (10953713, 4)
labevents: (84605867, 4)
Long structured rows: 95559580
Unique hadm_id with structured data: 448333


,hadm_id,time,variable,value
0,29079034,2180-07-23 17:00:00,input_226452,200.000000
1,29079034,2180-07-23 17:00:00,input_220862,49.999999
2,29079034,2180-07-23 17:33:00,input_220862,49.999999
3,29079034,2180-07-23 18:56:00,input_226452,100.000000
4,29079034,2180-07-23 21:10:00,input_226452,100.000000


## 4. Build optional note table

In [6]:
note_parts = []

for fname in NOTE_FILES:
    path = NOTE_ROOT / fname
    if not path.exists():
        print("Skipping missing note file:", path)
        continue

    df = pd.read_csv(path, low_memory=False)
    if "hadm_id" not in df.columns:
        print("Skipping note file without hadm_id:", fname)
        continue

    df["hadm_id"] = pd.to_numeric(df["hadm_id"], errors="coerce")
    df = df.dropna(subset=["hadm_id"]).copy()
    df["hadm_id"] = df["hadm_id"].astype(int)
    df = df.loc[df["hadm_id"].isin(ts_long["hadm_id"].unique())].copy()

    text_col = "text" if "text" in df.columns else None
    time_col = None
    for c in ["charttime", "storetime", "chartdate"]:
        if c in df.columns:
            time_col = c
            break

    if text_col is None:
        print("Skipping note file without text column:", fname)
        continue

    keep = ["hadm_id", text_col]
    if time_col is not None:
        keep.append(time_col)

    df = df[keep].copy()
    df = df.rename(columns={text_col: "text"})
    if time_col is not None:
        df = df.rename(columns={time_col: "note_time"})
        df["note_time"] = pd.to_datetime(df["note_time"], errors="coerce")
    else:
        df["note_time"] = pd.NaT

    df["text"] = df["text"].astype(str)
    note_parts.append(df)

if len(note_parts) > 0:
    notes = pd.concat(note_parts, ignore_index=True)
else:
    notes = pd.DataFrame(columns=["hadm_id", "note_time", "text"])

print("Note rows:", len(notes))
print("Unique hadm_id with any notes:", notes["hadm_id"].nunique() if len(notes) else 0)

notes.head()

Note rows: 1395433
Unique hadm_id with any notes: 327147


,hadm_id,text,note_time
0,22595853,EXAMINATION: CHEST (PA AND LAT)\n\nINDICATION...,2180-05-06 21:19:00
1,22595853,EXAMINATION: LIVER OR GALLBLADDER US (SINGLE ...,2180-05-06 23:00:00
2,22595853,"INDICATION: ___ HCV cirrhosis c/b ascites, hi...",2180-05-07 09:55:00
3,22841357,EXAMINATION: LIVER OR GALLBLADDER US (SINGLE ...,2180-06-26 17:15:00
4,22841357,EXAMINATION: CHEST (PA AND LAT)\n\nINDICATION...,2180-06-26 17:17:00


## 5. Minimum irregular-history rule

In [7]:
def summarize_irregular_history(ts_long_df, obs_hours):
    anchor = (
        ts_long_df.groupby("hadm_id")["time"]
        .min()
        .rename("anchor_time")
        .reset_index()
    )

    tmp = ts_long_df.merge(anchor, on="hadm_id", how="left")
    tmp["hours_from_anchor"] = (tmp["time"] - tmp["anchor_time"]) / pd.Timedelta(hours=1)
    tmp = tmp.loc[
        tmp["hours_from_anchor"].notna()
        & (tmp["hours_from_anchor"] >= 0)
        & (tmp["hours_from_anchor"] <= obs_hours)
    ].copy()

    summary = (
        tmp.groupby("hadm_id")
        .agg(
            n_unique_times=("time", "nunique"),
            n_obs_values=("value", lambda s: int(s.notna().sum())),
            n_distinct_variables=("variable", "nunique"),
            first_time=("time", "min"),
            last_time=("time", "max"),
        )
        .reset_index()
    )
    summary["obs_span_hours"] = (summary["last_time"] - summary["first_time"]) / pd.Timedelta(hours=1)
    return tmp, summary

ts_mort_early, mort_hist = summarize_irregular_history(ts_long, MORTALITY_OBS_HOURS)
ts_readm_early, readm_hist = summarize_irregular_history(ts_long, READMIT_OBS_HOURS)

mort_eligible = mort_hist.loc[
    (mort_hist["n_unique_times"] >= MIN_UNIQUE_TIMES) &
    (mort_hist["n_obs_values"] >= MIN_OBS_VALUES) &
    (mort_hist["n_distinct_variables"] >= MIN_DISTINCT_VARIABLES),
    "hadm_id"
].astype(int).tolist()

readm_eligible = readm_hist.loc[
    (readm_hist["n_unique_times"] >= MIN_UNIQUE_TIMES) &
    (readm_hist["n_obs_values"] >= MIN_OBS_VALUES) &
    (readm_hist["n_distinct_variables"] >= MIN_DISTINCT_VARIABLES),
    "hadm_id"
].astype(int).tolist()

# For readmission, must also be label-eligible
readm_label_eligible = set(
    label_df.loc[
        label_df["dischtime"].notna() & (label_df["mortality_label"] == 0),
        "hadm_id"
    ].astype(int).tolist()
)
readm_eligible = sorted(set(readm_eligible).intersection(readm_label_eligible))
mort_eligible = sorted(set(mort_eligible))

print("Mortality eligible:", len(mort_eligible))
print("Readmission eligible:", len(readm_eligible))

mort_hist.describe()

Mortality eligible: 62450
Readmission eligible: 79095


,hadm_id,n_unique_times,n_obs_values,n_distinct_variables,first_time,last_time,obs_span_hours
count,4.483330e+05,448333.000000,448333.000000,448333.000000,448333,448333,448333.000000
mean,2.499688e+07,5.693924,52.259834,38.222669,2155-02-13 04:56:06.726786048,2155-02-13 17:04:11.864751104,12.134761
min,2.000002e+07,1.000000,0.000000,1.000000,2105-10-04 17:41:00,2105-10-05 14:07:00,0.000000
25%,2.249139e+07,2.000000,25.000000,24.000000,2135-03-06 20:14:00,2135-03-07 05:45:00,0.066667
50%,2.499776e+07,2.000000,41.000000,34.000000,2154-12-24 06:25:00,2154-12-24 11:03:00,13.350000
75%,2.749939e+07,4.000000,63.000000,50.000000,2175-02-17 07:20:00,2175-02-17 17:54:00,21.583333
max,2.999994e+07,161.000000,699.000000,203.000000,2214-12-16 06:55:00,2214-12-17 06:50:00,24.000000
std,2.889349e+06,10.679808,46.529514,20.869893,NaN,NaN,9.295657


## 6. Helper functions to save folders

In [8]:
def pivot_timeseries(ts_df):
    wide = (
        ts_df.pivot_table(
            index="time",
            columns="variable",
            values="value",
            aggfunc="mean"
        )
        .reset_index()
        .sort_values("time")
    )
    return wide

def build_note_table(notes_df, hadm_id, anchor_time, obs_hours):
    g = notes_df.loc[notes_df["hadm_id"] == hadm_id].copy()
    if len(g) == 0:
        return pd.DataFrame(columns=["record_id", "date_time", "text"])

    g["date_time"] = pd.to_datetime(g["note_time"], errors="coerce")
    g = g.loc[g["date_time"].notna()].copy()

    if pd.notna(anchor_time):
        g["hours_from_anchor"] = (g["date_time"] - anchor_time) / pd.Timedelta(hours=1)
        g = g.loc[
            g["hours_from_anchor"].notna()
            & (g["hours_from_anchor"] >= 0)
            & (g["hours_from_anchor"] <= obs_hours)
        ].copy()

    if len(g) == 0:
        return pd.DataFrame(columns=["record_id", "date_time", "text"])

    out = g[["date_time", "text"]].copy()
    out.insert(0, "record_id", hadm_id)
    out = out.sort_values("date_time")
    return out

def write_task_folders(task_name, eligible_ids, early_ts_df, obs_hours):
    task_root_raw = OUT_BASE / f"raw_{task_name}"
    task_root_processed = OUT_BASE / f"processed_{task_name}"

    if task_root_raw.exists():
        shutil.rmtree(task_root_raw)
    if task_root_processed.exists():
        shutil.rmtree(task_root_processed)

    task_root_raw.mkdir(parents=True, exist_ok=True)
    task_root_processed.mkdir(parents=True, exist_ok=True)

    early_ts_df = early_ts_df.loc[early_ts_df["hadm_id"].isin(eligible_ids)].copy()

    anchor_df = (
        early_ts_df.groupby("hadm_id")["time"]
        .min()
        .rename("anchor_time")
        .reset_index()
    )

    stats = []

    for hadm_id, g in early_ts_df.groupby("hadm_id"):
        hadm_id = int(hadm_id)
        folder_raw = task_root_raw / str(hadm_id)
        folder_processed = task_root_processed / str(hadm_id)
        folder_raw.mkdir(parents=True, exist_ok=True)
        folder_processed.mkdir(parents=True, exist_ok=True)

        ts_wide = pivot_timeseries(g)
        if len(ts_wide) == 0:
            continue

        ts_out = ts_wide.copy()
        ts_out = ts_out.rename(columns={"time": "date_time"})
        ts_out.insert(0, "record_id", hadm_id)

        ts_out.to_csv(folder_raw / "time_series.csv", index=False)
        ts_out.to_csv(folder_processed / "time_series.csv", index=False)

        anchor_time = anchor_df.loc[anchor_df["hadm_id"] == hadm_id, "anchor_time"].iloc[0]
        text_out = build_note_table(notes, hadm_id, anchor_time, obs_hours)

        if len(text_out) == 0:
            text_out = pd.DataFrame(columns=["record_id", "date_time", "text"])

        text_out.to_csv(folder_raw / "text.csv", index=False)
        text_out.to_csv(folder_processed / "text.csv", index=False)

        stats.append({
            "hadm_id": hadm_id,
            "time_series_rows": len(ts_out),
            "text_rows": len(text_out),
            "n_features_present": max(0, ts_out.shape[1] - 2),
        })

    stats_df = pd.DataFrame(stats).sort_values("hadm_id")
    stats_df.to_csv(RUNLOG_DIR / f"{task_name}_folder_audit.csv", index=False)

    print(f"{task_name}: wrote {len(stats_df)} folders")
    print("raw path      :", task_root_raw)
    print("processed path:", task_root_processed)
    return stats_df

## 7. Write mortality folders

In [ ]:
mortality_stats = write_task_folders(
    task_name="mortality",
    eligible_ids=mort_eligible,
    early_ts_df=ts_mort_early,
    obs_hours=MORTALITY_OBS_HOURS,
)

mortality_stats.head()

## 8. Write ICU readmission folders

In [ ]:
readmit_stats = write_task_folders(
    task_name="icu_readmit",
    eligible_ids=readm_eligible,
    early_ts_df=ts_readm_early,
    obs_hours=READMIT_OBS_HOURS,
)

readmit_stats.head()

## 9. Save task-specific label files

In [ ]:
mortality_labels = label_df.loc[label_df["hadm_id"].isin(mort_eligible), ["hadm_id", "mortality_label"]].copy()
readmit_labels   = label_df.loc[label_df["hadm_id"].isin(readm_eligible), ["hadm_id", "readmit_label"]].copy()

label_out_dir = OUT_BASE / "labels"
label_out_dir.mkdir(parents=True, exist_ok=True)

mortality_labels.to_csv(label_out_dir / "mortality_labels.csv", index=False)
readmit_labels.to_csv(label_out_dir / "icu_readmit_labels.csv", index=False)

mort_hist.to_csv(RUNLOG_DIR / "mortality_irregular_history_summary.csv", index=False)
readm_hist.to_csv(RUNLOG_DIR / "icu_readmit_irregular_history_summary.csv", index=False)

print("Saved labels to:", label_out_dir)
print("Mortality positives:", mortality_labels["mortality_label"].sum(), "of", len(mortality_labels))
print("Readmit positives  :", readmit_labels["readmit_label"].sum(), "of", len(readmit_labels))

## 10. Final audit

In [ ]:
def count_folders(path):
    if not Path(path).exists():
        return 0
    return len([x for x in Path(path).iterdir() if x.is_dir()])

summary = pd.DataFrame([
    {
        "task": "mortality",
        "eligible_ids": len(mort_eligible),
        "raw_folders": count_folders(OUT_BASE / "raw_mortality"),
        "processed_folders": count_folders(OUT_BASE / "processed_mortality"),
        "positive_rate": mortality_labels["mortality_label"].mean() if len(mortality_labels) else np.nan,
    },
    {
        "task": "icu_readmit",
        "eligible_ids": len(readm_eligible),
        "raw_folders": count_folders(OUT_BASE / "raw_icu_readmit"),
        "processed_folders": count_folders(OUT_BASE / "processed_icu_readmit"),
        "positive_rate": readmit_labels["readmit_label"].mean() if len(readmit_labels) else np.nan,
    }
])

summary.to_csv(RUNLOG_DIR / "final_mimic_preprocess_summary.csv", index=False)
summary

## 11. Where to point your downstream runners

Use these directories:

- mortality processed folder:
  `"/content/drive/MyDrive/IMM-TSF/data/MIMIC_fixed/processed_mortality"`
- ICU readmission processed folder:
  `"/content/drive/MyDrive/IMM-TSF/data/MIMIC_fixed/processed_icu_readmit"`

Use these labels:

- mortality:
  `"/content/drive/MyDrive/IMM-TSF/data/MIMIC_fixed/labels/mortality_labels.csv"`
- ICU readmission:
  `"/content/drive/MyDrive/IMM-TSF/data/MIMIC_fixed/labels/icu_readmit_labels.csv"`